In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — COPULA-BASED TAIL DEPENDENCE
# Inputs:  outputs/02_returns_matrix.parquet
#          outputs/phase1_fat_tail_params.csv
#          outputs/phase2_clustering_baseline.csv
#          outputs/03_coin_metadata.csv
# Outputs: outputs/06_tail_dependence_lower.parquet   ← λL matrix (crash)
#          outputs/07_tail_dependence_upper.parquet   ← λU matrix (rally)
#          outputs/copula_aic_best.csv                ← best-fit copula per pair
#          outputs/phase3_tail_dependence_pairs.csv   ← full pair-level results
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

from scipy.stats import norm
from scipy.optimize import minimize_scalar, minimize
from joblib import Parallel, delayed

Path("outputs").mkdir(exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD INPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("Loading Phase 2 outputs...")

log_returns = pd.read_parquet("outputs/02_returns_matrix.parquet")
fat_tail_df = pd.read_csv("outputs/phase1_fat_tail_params.csv")
cluster_df  = pd.read_csv("outputs/phase2_clustering_baseline.csv")
coin_meta   = pd.read_csv("outputs/03_coin_metadata.csv")

coins = log_returns.columns.tolist()
N     = len(coins)

# Extreme fat-tail coin set — Gaussian copula forbidden for these
extreme_coins = set(
    fat_tail_df.loc[fat_tail_df["extreme_fat_tail"] == True, "coin_id"].tolist()
)

print(f"  Coins                  : {N}")
print(f"  Extreme fat-tail coins : {len(extreme_coins)}  (Gaussian copula excluded)")
print(f"  Total pairs            : {N*(N-1)//2:,}")


# ═══════════════════════════════════════════════════════════════════════════════
# PROBABILITY INTEGRAL TRANSFORM (PIT)
# ═══════════════════════════════════════════════════════════════════════════════
# Transform each coin's return series to uniform marginals u ∈ (0,1).
# This isolates the DEPENDENCE STRUCTURE from the marginal distributions.
# Why this matters: copulas model how two variables move together independent
# of how each is individually distributed. By transforming to uniforms first,
# we ensure our tail dependence estimates reflect co-movement structure, not
# the shape of individual return distributions.
#
# We use the empirical CDF (rank-based) rather than a parametric CDF.
# Reason: even with Student-t marginals fitted in Phase 1.2, the empirical
# CDF is more robust and makes no parametric assumptions about the marginals.

def pit_transform(series: pd.Series) -> pd.Series:
    """
    Probability Integral Transform via empirical CDF.
    Ranks normalised to (0,1) open interval: rank / (n+1).
    Avoids boundary values 0 and 1 which cause log(0) in copula likelihoods.
    """
    clean = series.dropna()
    n     = len(clean)
    u     = clean.rank() / (n + 1)
    return u


print("\nApplying PIT transform to all coins...")
pit_df = pd.DataFrame(index=log_returns.index)
for coin in coins:
    u = pit_transform(log_returns[coin])
    pit_df[coin] = np.nan
    pit_df.loc[u.index, coin] = u.values

print(f"  PIT matrix shape : {pit_df.shape}")
print(f"  Value range      : [{pit_df.min().min():.4f}, {pit_df.max().max():.4f}]  (expected: near 0 to 1)")


# ═══════════════════════════════════════════════════════════════════════════════
# COPULA LOG-LIKELIHOOD FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

EPS = 1e-10   # numerical stability floor

def _safe_uv(u: np.ndarray, v: np.ndarray):
    """Clip u, v away from 0 and 1 for numerical stability."""
    return np.clip(u, EPS, 1 - EPS), np.clip(v, EPS, 1 - EPS)


# ── 1. Gaussian Copula ────────────────────────────────────────────────────────
# Symmetric — no tail dependence (λL = λU = 0).
# Baseline model. We exclude this for pairs where either coin has ν < 4
# since the Gaussian copula structurally cannot capture the tail co-movement
# that extreme fat-tail coins exhibit.

def gaussian_copula_loglik(rho: float, u: np.ndarray, v: np.ndarray) -> float:
    if abs(rho) >= 1.0:
        return -np.inf
    u, v   = _safe_uv(u, v)
    x, y   = norm.ppf(u), norm.ppf(v)
    r2     = 1 - rho**2
    # Log-density of bivariate normal minus standard normal marginals
    log_c  = (-0.5 * np.log(r2)
               - 0.5 / r2 * (x**2 + y**2 - 2 * rho * x * y)
               + 0.5 * (x**2 + y**2))
    ll     = np.sum(log_c[np.isfinite(log_c)])
    return ll if np.isfinite(ll) else -np.inf


def fit_gaussian(u: np.ndarray, v: np.ndarray) -> dict:
    res = minimize_scalar(
        lambda r: -gaussian_copula_loglik(r, u, v),
        bounds=(-0.999, 0.999), method="bounded"
    )
    rho    = res.x
    ll     = -res.fun
    k      = 1
    n      = len(u)
    aic    = 2 * k - 2 * ll
    bic    = k * np.log(n) - 2 * ll
    return {"model": "gaussian", "param1": rho, "param2": np.nan,
            "loglik": ll, "aic": aic, "bic": bic,
            "lambda_L": 0.0, "lambda_U": 0.0}


# ── 2. Clayton Copula ─────────────────────────────────────────────────────────
# Lower tail dependence only: λL = 2^(-1/θ), λU = 0.
# PRIMARY model for crash co-movement. θ > 0.
# The higher θ, the more the two assets tend to crash together simultaneously.

def clayton_copula_loglik(theta: float, u: np.ndarray, v: np.ndarray) -> float:
    if theta <= 0:
        return -np.inf
    u, v   = _safe_uv(u, v)
    n      = len(u)
    log_c  = (np.log(1 + theta)
              - (1 + theta) * (np.log(u) + np.log(v))
              - (2 + 1 / theta) * np.log(u**(-theta) + v**(-theta) - 1))
    mask   = np.isfinite(log_c)
    ll     = np.sum(log_c[mask])
    return ll if np.isfinite(ll) else -np.inf


def fit_clayton(u: np.ndarray, v: np.ndarray) -> dict:
    res = minimize_scalar(
        lambda t: -clayton_copula_loglik(t, u, v),
        bounds=(1e-4, 20.0), method="bounded"
    )
    theta  = res.x
    ll     = -res.fun
    k, n   = 1, len(u)
    aic    = 2 * k - 2 * ll
    bic    = k * np.log(n) - 2 * ll
    lam_L  = 2 ** (-1 / theta) if theta > 0 else 0.0
    return {"model": "clayton", "param1": theta, "param2": np.nan,
            "loglik": ll, "aic": aic, "bic": bic,
            "lambda_L": lam_L, "lambda_U": 0.0}


# ── 3. Gumbel Copula ──────────────────────────────────────────────────────────
# Upper tail dependence only: λU = 2 - 2^(1/θ), λL = 0. θ ≥ 1.
# Captures RALLY co-movement — do these coins pump together?
# At θ = 1: independence. As θ → ∞: perfect upper tail dependence.

def gumbel_copula_loglik(theta: float, u: np.ndarray, v: np.ndarray) -> float:
    if theta < 1.0:
        return -np.inf
    u, v      = _safe_uv(u, v)
    x, y      = -np.log(u), -np.log(v)          # x, y > 0
    A         = x**theta + y**theta
    A_pow     = A ** (1 / theta)                  # A^(1/θ)
    # C(u,v) = exp(-A^(1/θ))
    # log density: -A^(1/θ) + (θ-1)*log(xy) + (1/θ-2)*log(A)
    #              + log(1 + (θ-1)*A^(-1/θ)) - log(uv)
    log_c     = (-A_pow
                 + (theta - 1) * np.log(x * y)
                 + (1 / theta - 2) * np.log(A)
                 + np.log(1 + (theta - 1) / A_pow)
                 - np.log(u * v))
    mask      = np.isfinite(log_c)
    ll        = np.sum(log_c[mask])
    return ll if np.isfinite(ll) else -np.inf


def fit_gumbel(u: np.ndarray, v: np.ndarray) -> dict:
    res = minimize_scalar(
        lambda t: -gumbel_copula_loglik(t, u, v),
        bounds=(1.0001, 20.0), method="bounded"
    )
    theta  = res.x
    ll     = -res.fun
    k, n   = 1, len(u)
    aic    = 2 * k - 2 * ll
    bic    = k * np.log(n) - 2 * ll
    lam_U  = 2 - 2 ** (1 / theta) if theta > 1 else 0.0
    return {"model": "gumbel", "param1": theta, "param2": np.nan,
            "loglik": ll, "aic": aic, "bic": bic,
            "lambda_L": 0.0, "lambda_U": lam_U}


# ── 4. Frank Copula ───────────────────────────────────────────────────────────
# Symmetric, NO tail dependence (λL = λU = 0). θ ∈ ℝ \ {0}.
# Captures symmetric dependence — good for pairs where co-movement is
# neither crash-driven nor rally-driven. Acts as a null model for tail dependence:
# if Frank fits best, the pair has no meaningful tail co-movement.

def frank_copula_loglik(theta: float, u: np.ndarray, v: np.ndarray) -> float:
    if abs(theta) < 1e-6:
        return -np.inf
    u, v   = _safe_uv(u, v)
    et     = np.exp(-theta)
    eu     = np.exp(-theta * u)
    ev     = np.exp(-theta * v)
    denom  = (et - 1) + (eu - 1) * (ev - 1)
    denom  = np.where(np.abs(denom) < EPS, EPS, denom)
    log_c  = (np.log(abs(theta)) + np.log(abs(et - 1))
              - theta * (u + v)
              - 2 * np.log(np.abs(denom)))
    mask   = np.isfinite(log_c)
    ll     = np.sum(log_c[mask])
    return ll if np.isfinite(ll) else -np.inf


def fit_frank(u: np.ndarray, v: np.ndarray) -> dict:
    # Search over positive and negative theta
    best_ll, best_theta = -np.inf, 1.0
    for t0 in [-10, -5, -1, 1, 5, 10]:
        res = minimize(
            lambda t: -frank_copula_loglik(t[0], u, v),
            x0=[t0], method="Nelder-Mead",
            options={"xatol": 1e-4, "fatol": 1e-4, "maxiter": 500}
        )
        if -res.fun > best_ll:
            best_ll, best_theta = -res.fun, res.x[0]
    k, n   = 1, len(u)
    aic    = 2 * k - 2 * best_ll
    bic    = k * np.log(n) - 2 * best_ll
    return {"model": "frank", "param1": best_theta, "param2": np.nan,
            "loglik": best_ll, "aic": aic, "bic": bic,
            "lambda_L": 0.0, "lambda_U": 0.0}


# ── Non-parametric tail dependence estimator ─────────────────────────────────
# Model-free λL and λU via empirical conditional probabilities.
# λL ≈ P(V < q | U < q) for q → 0 (we use q = 0.10, 0.15, 0.20 and average)
# This is our ROBUSTNESS CHECK against the parametric estimates.
# If parametric Clayton λL and non-parametric λL agree → high confidence.

def nonparametric_tail_dependence(u: np.ndarray, v: np.ndarray,
                                   quantiles=(0.10, 0.15, 0.20)) -> dict:
    lam_L_vals, lam_U_vals = [], []
    for q in quantiles:
        # Lower tail
        mask_L = u < q
        lam_L  = np.mean(v[mask_L] < q) if mask_L.sum() >= 5 else np.nan
        lam_L_vals.append(lam_L)
        # Upper tail
        mask_U = u > (1 - q)
        lam_U  = np.mean(v[mask_U] > (1 - q)) if mask_U.sum() >= 5 else np.nan
        lam_U_vals.append(lam_U)
    return {
        "np_lambda_L": np.nanmean(lam_L_vals),
        "np_lambda_U": np.nanmean(lam_U_vals),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# AIC-BASED MODEL SELECTION
# ═══════════════════════════════════════════════════════════════════════════════
def select_best_copula(results: list) -> dict:
    """Pick the copula with lowest AIC. Return best model dict."""
    valid = [r for r in results if r is not None and np.isfinite(r["aic"])]
    if not valid:
        return None
    return min(valid, key=lambda r: r["aic"])


# ═══════════════════════════════════════════════════════════════════════════════
# PER-PAIR FITTING FUNCTION  (called in parallel)
# ═══════════════════════════════════════════════════════════════════════════════
def fit_pair(args):
    """
    Fits all copula models for a single (coin_i, coin_j) pair.
    Returns a flat dict of results or None if insufficient data.
    """
    i, j, coin_i, coin_j, extreme_set = args

    ui = pit_df[coin_i].dropna()
    uj = pit_df[coin_j].dropna()

    # Align on common dates
    common  = ui.index.intersection(uj.index)
    if len(common) < 60:
        return None

    u = ui.loc[common].values.astype(float)
    v = uj.loc[common].values.astype(float)

    # Remove any remaining NaNs
    mask = np.isfinite(u) & np.isfinite(v)
    u, v = u[mask], v[mask]
    if len(u) < 60:
        return None

    # Determine which copulas to fit
    # Gaussian excluded if either coin has extreme fat tails
    use_gaussian = (coin_i not in extreme_set) and (coin_j not in extreme_set)

    results = []
    if use_gaussian:
        try:
            results.append(fit_gaussian(u, v))
        except Exception:
            pass

    try:
        results.append(fit_clayton(u, v))
    except Exception:
        pass

    try:
        results.append(fit_gumbel(u, v))
    except Exception:
        pass

    try:
        results.append(fit_frank(u, v))
    except Exception:
        pass

    best = select_best_copula(results)
    np_td = nonparametric_tail_dependence(u, v)

    # Extract per-model λL and λU for output
    model_lam = {r["model"]: r for r in results if r is not None}

    row = {
        "coin_a"            : coin_i,
        "coin_b"            : coin_j,
        "n_obs"             : len(u),
        # Best model
        "best_model"        : best["model"]    if best else "none",
        "best_aic"          : best["aic"]      if best else np.nan,
        "best_lambda_L"     : best["lambda_L"] if best else np.nan,
        "best_lambda_U"     : best["lambda_U"] if best else np.nan,
        # Clayton (primary crash measure)
        "clayton_theta"     : model_lam.get("clayton", {}).get("param1", np.nan),
        "clayton_lambda_L"  : model_lam.get("clayton", {}).get("lambda_L", np.nan),
        "clayton_aic"       : model_lam.get("clayton", {}).get("aic", np.nan),
        # Gumbel (rally measure)
        "gumbel_theta"      : model_lam.get("gumbel", {}).get("param1", np.nan),
        "gumbel_lambda_U"   : model_lam.get("gumbel", {}).get("lambda_U", np.nan),
        "gumbel_aic"        : model_lam.get("gumbel", {}).get("aic", np.nan),
        # Gaussian (baseline)
        "gaussian_rho"      : model_lam.get("gaussian", {}).get("param1", np.nan),
        "gaussian_aic"      : model_lam.get("gaussian", {}).get("aic", np.nan),
        # Frank (no-tail null)
        "frank_theta"       : model_lam.get("frank", {}).get("param1", np.nan),
        "frank_aic"         : model_lam.get("frank", {}).get("aic", np.nan),
        # Non-parametric estimates
        "np_lambda_L"       : np_td["np_lambda_L"],
        "np_lambda_U"       : np_td["np_lambda_U"],
        # Flags
        "gaussian_excluded" : not use_gaussian,
    }
    return row


# ═══════════════════════════════════════════════════════════════════════════════
# 3.1  VALIDATION RUN — TOP 50 MOST LIQUID COINS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 3.1 │ Validation Run — Top 50 Liquid Coins ───────────────────────")

# Sort by median daily volume descending — take top 50
top50_coins = (
    coin_meta.sort_values("median_daily_vol", ascending=False)
    .head(50)["coin_id"]
    .tolist()
)
# Keep only those present in return matrix
top50_coins = [c for c in top50_coins if c in coins]
print(f"  Top 50 liquid coins selected : {len(top50_coins)}")

# Build pair list for top 50
top50_pairs = [
    (coins.index(ci), coins.index(cj), ci, cj, extreme_coins)
    for ci, cj in combinations(top50_coins, 2)
    if ci in coins and cj in coins
]
print(f"  Pairs to fit                 : {len(top50_pairs):,}")
print(f"  Running copula fits (parallel)...")

val_results = Parallel(n_jobs=-1, verbose=0)(
    delayed(fit_pair)(args) for args in top50_pairs
)
val_results = [r for r in val_results if r is not None]
val_df      = pd.DataFrame(val_results)

# ─── Validation summary ───────────────────────────────────────────────────────
print(f"\n  Validation results ({len(val_df):,} pairs):")
print(f"    Best model distribution:")
for model, cnt in val_df["best_model"].value_counts().items():
    pct = cnt / len(val_df) * 100
    print(f"      {model:>10} : {cnt:4d} pairs  ({pct:.1f}%)")

# Sanity check: BTC–ETH λL
if "bitcoin" in top50_coins and "ethereum" in top50_coins:
    btc_eth = val_df[
        ((val_df["coin_a"] == "bitcoin") & (val_df["coin_b"] == "ethereum")) |
        ((val_df["coin_a"] == "ethereum") & (val_df["coin_b"] == "bitcoin"))
    ]
    if len(btc_eth):
        row = btc_eth.iloc[0]
        print(f"\n  Sanity check — BTC–ETH:")
        print(f"    Best model      : {row['best_model']}")
        print(f"    Clayton λL      : {row['clayton_lambda_L']:.4f}  (expected > 0.30)")
        print(f"    Gumbel  λU      : {row['gumbel_lambda_U']:.4f}")
        print(f"    Non-param λL    : {row['np_lambda_L']:.4f}")

# λL distribution
print(f"\n  Clayton λL distribution (top 50 pairs):")
lL = val_df["clayton_lambda_L"].dropna()
print(f"    Mean   : {lL.mean():.4f}")
print(f"    Median : {lL.median():.4f}")
print(f"    Pairs λL > 0.50 (high crash similarity) : {(lL > 0.50).sum()}")
print(f"    Pairs λL > 0.30 (moderate)              : {(lL > 0.30).sum()}")
print(f"    Pairs λL < 0.30 (low crash similarity)  : {(lL < 0.30).sum()}")


# ═══════════════════════════════════════════════════════════════════════════════
# 3.2  FULL UNIVERSE RUN — ALL 7,750 PAIRS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 3.2 │ Full Universe Run — All Pairs ───────────────────────────────")

all_pairs = [
    (coins.index(ci), coins.index(cj), ci, cj, extreme_coins)
    for ci, cj in combinations(coins, 2)
]
print(f"  Total pairs : {len(all_pairs):,}")
print(f"  Running parallel copula fits — this will take several minutes...")

all_results = Parallel(n_jobs=-1, verbose=10)(
    delayed(fit_pair)(args) for args in all_pairs
)
all_results = [r for r in all_results if r is not None]
results_df  = pd.DataFrame(all_results)

print(f"\n  Completed : {len(results_df):,} pairs fitted")
print(f"  Failed    : {len(all_pairs) - len(results_df):,} pairs (insufficient data)")

# ─── Full universe summary ────────────────────────────────────────────────────
print(f"\n  Best model distribution (full universe):")
for model, cnt in results_df["best_model"].value_counts().items():
    pct = cnt / len(results_df) * 100
    print(f"    {model:>10} : {cnt:5d} pairs  ({pct:.1f}%)")

lL_full = results_df["clayton_lambda_L"].dropna()
lU_full = results_df["gumbel_lambda_U"].dropna()
np_lL   = results_df["np_lambda_L"].dropna()

print(f"\n  Clayton λL distribution (crash similarity):")
print(f"    Mean   : {lL_full.mean():.4f}")
print(f"    Median : {lL_full.median():.4f}")
print(f"    Std    : {lL_full.std():.4f}")
print(f"    Pairs λL > 0.50  (high)     : {(lL_full > 0.50).sum():,}")
print(f"    Pairs λL 0.30–0.50 (moderate): {((lL_full >= 0.30) & (lL_full <= 0.50)).sum():,}")
print(f"    Pairs λL < 0.30  (low)      : {(lL_full < 0.30).sum():,}")

print(f"\n  Gumbel λU distribution (rally similarity):")
print(f"    Mean   : {lU_full.mean():.4f}")
print(f"    Median : {lU_full.median():.4f}")
print(f"    Pairs λU > 0.50  (high rally co-movement) : {(lU_full > 0.50).sum():,}")

print(f"\n  Parametric vs Non-parametric λL agreement:")
common_idx = results_df[["clayton_lambda_L", "np_lambda_L"]].dropna().index
if len(common_idx) > 0:
    param_vals = results_df.loc[common_idx, "clayton_lambda_L"]
    np_vals    = results_df.loc[common_idx, "np_lambda_L"]
    from scipy.stats import spearmanr
    corr, _ = spearmanr(param_vals, np_vals)
    mean_diff = (param_vals - np_vals).abs().mean()
    print(f"    Spearman rank correlation : {corr:.4f}  (> 0.70 = good agreement)")
    print(f"    Mean absolute difference  : {mean_diff:.4f}")


# ═══════════════════════════════════════════════════════════════════════════════
# 3.3  TAIL DEPENDENCE CLASSIFICATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 3.3 │ Tail Dependence Classification ──────────────────────────────")

def classify_tail(lam_L: float) -> str:
    if   lam_L > 0.50: return "HIGH_CRASH_SIMILARITY"
    elif lam_L > 0.30: return "MODERATE"
    else:              return "LOW_CRASH_SIMILARITY"

results_df["crash_similarity_class"] = results_df["clayton_lambda_L"].apply(
    lambda x: classify_tail(x) if not np.isnan(x) else "UNKNOWN"
)

class_counts = results_df["crash_similarity_class"].value_counts()
print(f"\n  Crash similarity classification:")
for cls, cnt in class_counts.items():
    pct = cnt / len(results_df) * 100
    print(f"    {cls:30s} : {cnt:5,} pairs  ({pct:.1f}%)")

# Top 20 highest crash similarity pairs
print(f"\n  Top 20 highest crash similarity pairs (λL):")
top20 = (results_df
         .sort_values("clayton_lambda_L", ascending=False)
         .head(20)[["coin_a", "coin_b", "clayton_lambda_L",
                     "gumbel_lambda_U", "np_lambda_L", "best_model"]])
print(top20.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD λL AND λU MATRICES (N × N)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Building tail dependence matrices ───────────────────────────────────────")

lam_L_arr = np.full((N, N), np.nan)
lam_U_arr = np.full((N, N), np.nan)
np.fill_diagonal(lam_L_arr, 1.0)
np.fill_diagonal(lam_U_arr, 1.0)
lam_L_mat = pd.DataFrame(lam_L_arr, index=coins, columns=coins)
lam_U_mat = pd.DataFrame(lam_U_arr, index=coins, columns=coins)

for _, row in results_df.iterrows():
    ca, cb = row["coin_a"], row["coin_b"]
    if ca in coins and cb in coins:
        lam_L_mat.loc[ca, cb] = lam_L_mat.loc[cb, ca] = row["clayton_lambda_L"]
        lam_U_mat.loc[ca, cb] = lam_U_mat.loc[cb, ca] = row["gumbel_lambda_U"]

nan_pct_L = lam_L_mat.isna().mean().mean()
nan_pct_U = lam_U_mat.isna().mean().mean()
print(f"  λL matrix NaN rate : {nan_pct_L:.2%}")
print(f"  λU matrix NaN rate : {nan_pct_U:.2%}")


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving Phase 3 outputs ──────────────────────────────────────────────────")

lam_L_mat.to_parquet("outputs/06_tail_dependence_lower.parquet")
print("  ✅ 06_tail_dependence_lower.parquet  → λL (crash similarity) matrix")

lam_U_mat.to_parquet("outputs/07_tail_dependence_upper.parquet")
print("  ✅ 07_tail_dependence_upper.parquet  → λU (rally similarity) matrix")

results_df.to_csv("outputs/phase3_tail_dependence_pairs.csv", index=False)
print("  ✅ phase3_tail_dependence_pairs.csv  → full pair-level results")

# Best copula per pair
copula_aic = results_df[["coin_a", "coin_b", "best_model", "best_aic",
                          "clayton_lambda_L", "gumbel_lambda_U",
                          "np_lambda_L", "np_lambda_U",
                          "crash_similarity_class", "gaussian_excluded"]]
copula_aic.to_csv("outputs/copula_aic_best.csv", index=False)
print("  ✅ copula_aic_best.csv               → best-fit copula + λL/λU per pair")

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  PHASE 3 COMPLETE                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  Pairs fitted               : {len(results_df):,}                           ║
║  Median Clayton λL          : {lL_full.median():.4f}                           ║
║  Median Gumbel  λU          : {lU_full.median():.4f}                           ║
║  High crash similarity      : {(lL_full > 0.50).sum():,} pairs  (λL > 0.50)       ║
║  Gaussian excluded pairs    : {results_df['gaussian_excluded'].sum():,}                           ║
╠══════════════════════════════════════════════════════════════════╣
║  Next: Phase 4 — Regime-Conditional Similarity                  ║
║  Key inputs for Phase 4:                                         ║
║    02_returns_matrix.parquet                                     ║
║    06_tail_dependence_lower.parquet  ← λL for regime split      ║
║    subperiod_labels.csv              ← consolidated periods      ║
╚══════════════════════════════════════════════════════════════════╝
""")

Loading Phase 2 outputs...
  Coins                  : 125
  Extreme fat-tail coins : 56  (Gaussian copula excluded)
  Total pairs            : 7,750

Applying PIT transform to all coins...
  PIT matrix shape : (729, 125)
  Value range      : [0.0014, 0.9986]  (expected: near 0 to 1)

── Phase 3.1 │ Validation Run — Top 50 Liquid Coins ───────────────────────
  Top 50 liquid coins selected : 50
  Pairs to fit                 : 1,225
  Running copula fits (parallel)...
